# 01 - Preprocessing (MedQuAD)

Install packages, clone the repo, load MedQuAD directly from the HF Hub, tokenize, and verify. No training here.

In [ ]:
!pip install -q datasets transformers sentencepiece

In [ ]:
import os
import sys

REPO_URL = "https://github.com/satyazm/Finetuning_LLMs.git"
REPO_DIR = "/kaggle/working/Finetuning_LLMs"

if not os.path.exists(REPO_DIR):
    clone_url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass
    !git clone -q {clone_url} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)

## Load dataset

`preprocess.run` calls `datasets.load_dataset` directly against `keivalya/MedQuad-MedicalQnADataset` on the HF Hub, cleans/dedupes it, formats it as instruction/input/output records, splits it, and saves JSON.

In [ ]:
from src.data.preprocess import run

run(output_dir="/kaggle/working/data")

## Verify

In [ ]:
import json

with open("/kaggle/working/data/train.json") as f:
    train = json.load(f)
with open("/kaggle/working/data/val.json") as f:
    val = json.load(f)
with open("/kaggle/working/data/test.json") as f:
    test = json.load(f)

print(f"train={len(train)} val={len(val)} test={len(test)}")
print(json.dumps(train[0], indent=2))

## Tokenize

Use the base model's tokenizer (from `configs/model.yaml`) to check sequence-length distribution and pick `max_seq_length` for training.

In [ ]:
import yaml
from transformers import AutoTokenizer

with open("configs/model.yaml") as f:
    model_cfg = yaml.safe_load(f)

tokenizer = AutoTokenizer.from_pretrained(model_cfg["base_model"])


def to_text(example):
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"


lengths = sorted(len(tokenizer(to_text(ex))["input_ids"]) for ex in train)
n = len(lengths)
print(f"min={lengths[0]} mean={sum(lengths) // n} p50={lengths[n // 2]} p95={lengths[int(n * 0.95)]} max={lengths[-1]}")
print(f"configured max_seq_length={model_cfg['max_seq_length']}")

In [ ]:
sample = to_text(train[0])
encoded = tokenizer(sample)["input_ids"]
decoded = tokenizer.decode(encoded)
print(decoded[:400])